In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('../../'))

import numpy as np
import json, time
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import tensorflow as tf
from tensorflow import keras

from src.rnn.layers import SimpleRNNCell, LSTMCell, DenseProjection, DenseOutput
from src.rnn.forward_propagation import SimpleRNNCaptioner, LSTMCaptioner
from src.rnn.models import build_rnn_decoder_pre_inject, build_rnn_decoder_init_inject
from src.rnn.evaluate import compute_bleu4, get_references, keras_greedy_decode
from src.rnn.preprocessing import load_vocab, get_id2word
from src.rnn.train import masked_sparse_crossentropy

REPO       = os.path.abspath('../../')
WEIGHTS    = os.path.join(REPO, 'weights', 'rnn_lstm')
DATA       = os.path.join(REPO, 'data')
PLOTS      = os.path.join(REPO, 'results', 'plots')
TABLES     = os.path.join(REPO, 'results', 'tables')
os.makedirs(PLOTS, exist_ok=True)
os.makedirs(TABLES, exist_ok=True)

vocab    = load_vocab(os.path.join(DATA, 'captions', 'vocab.json'))
id2word  = get_id2word(vocab)
VOCAB_SZ = len(vocab)
print(f'Vocab size: {VOCAB_SZ}')

feat_dict  = np.load(os.path.join(DATA, 'features', 'test_features.npy'), allow_pickle=True).item()
test_names = list(feat_dict.keys())
test_feats = np.stack([feat_dict[n] for n in test_names]).astype(np.float32)
FEAT_DIM   = test_feats.shape[1]
print(f'Test images: {len(test_names)}, feat_dim: {FEAT_DIM}')

with open(os.path.join(DATA, 'captions', 'test_captions.json')) as f:
    test_caps = json.load(f)
all_refs = get_references(test_names, test_caps)

CUSTOM = {'masked_sparse_crossentropy': masked_sparse_crossentropy}
print('Setup selesai')

2026-05-15 11:24:48.050242: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778844288.072998    1783 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778844288.080381    1783 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778844288.098446    1783 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778844288.098467    1783 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778844288.098470    1783 computation_placer.cc:177] computation placer alr

Vocab size: 2576
Test images: 1012, feat_dim: 2048


Setup selesai


---
## Bonus 1 — Init-inject vs Pre-inject Architecture

**Pre-inject**: Image feature diproyeksikan ke embedding space dan di-*prepend* ke urutan kata sebagai token pertama, lalu seluruh sequence diproses RNN/LSTM.

**Init-inject**: Image feature diproyeksikan langsung ke *initial hidden state* (h₀, dan c₀ untuk LSTM). RNN/LSTM diinisialisasi dengan informasi gambar dan langsung memproses kata-kata tanpa image token tambahan.

In [2]:
print('=== PRE-INJECT (image sebagai token pertama) ===')
pre = build_rnn_decoder_pre_inject(
    vocab_size=VOCAB_SZ, embed_dim=256, rnn_units=256,
    feature_dim=FEAT_DIM, num_rnn_layers=2, rnn_type='lstm')
pre.summary()

print('\n=== INIT-INJECT (image sebagai initial hidden state) ===')
init = build_rnn_decoder_init_inject(
    vocab_size=VOCAB_SZ, embed_dim=256, rnn_units=256,
    feature_dim=FEAT_DIM, num_rnn_layers=2, rnn_type='lstm')
init.summary()

=== PRE-INJECT (image sebagai token pertama) ===


I0000 00:00:1778844298.807254    1783 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5447 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1778844298.812680    1783 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13655 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Model: "lstm_captioner_pre_inject"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ cnn_input           │ (None, 2048)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ image_projection    │ (None, 256)       │    524,544 │ cnn_input[0][0]   │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ word_input          │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ image_token         │ (None, 1, 256)    │          0 │ image_projection… │
│ (Reshape)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, None, 256) │    659,456 │ word_input[0][0]  │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concat_img_words    │ (None, None, 256) │          0 │ image_token[0][0… │
│ (Concatenate)       │                   │            │ embedding[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rnn_layer_0 (LSTM)  │ (None, None, 256) │    525,312 │ concat_img_words… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rnn_layer_1 (LSTM)  │ (None, None, 256) │    525,312 │ rnn_layer_0[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ output_dense        │ (None, None,      │    662,032 │ rnn_layer_1[0][0] │
│ (Dense)             │ 2576)             │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 2,896,656 (11.05 MB)

...

Model: "lstm_captioner_init_inject"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ word_input          │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cnn_input           │ (None, 2048)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, None, 256) │    659,456 │ word_input[0][0]  │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ h0_projection       │ (None, 256)       │    524,544 │ cnn_input[0][0]   │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ c0_projection_0     │ (None, 256)       │    524,544 │ cnn_input[0][0]   │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rnn_layer_0 (LSTM)  │ (None, None, 256) │    525,312 │ embedding[0][0],  │
│                     │                   │            │ h0_projection[0]… │
│                     │                   │            │ c0_projection_0[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ h0_projection_1     │ (None, 256)       │    524,544 │ cnn_input[0][0]   │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ c0_projection_1     │ (None, 256)       │    524,544 │ cnn_input[0][0]   │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rnn_layer_1 (LSTM)  │ (None, None, 256) │    525,312 │ rnn_layer_0[0][0… │
│                     │                   │            │ h0_projection_1[… │
│                     │                   │            │ c0_projection_1[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ output_dense        │ (None, None,      │    662,032 │ rnn_layer_1[0][0] │
│ (Dense)             │ 2576)             │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 4,470,288 (17.05 MB)

 Trainable params: 4,470,288 (17.05 MB)

 Non-trainable params: 0 (0.00 B)

In [3]:
def eval_keras(model_name, n=None):
    path = os.path.join(WEIGHTS, f'{model_name}.h5')
    if not os.path.exists(path):
        return None
    m = keras.models.load_model(path, compile=False, custom_objects=CUSTOM)
    feats = test_feats if n is None else test_feats[:n]
    refs  = all_refs  if n is None else all_refs[:n]
    t0 = time.perf_counter()
    hyps = keras_greedy_decode(m, feats, vocab, max_len=30)
    elapsed = time.perf_counter() - t0
    b4 = compute_bleu4(hyps, refs)
    return {'model': model_name, 'bleu4': round(b4, 4), 'time_s': round(elapsed, 2)}

print('=== Evaluasi BLEU-4: Pre-inject vs Init-inject ===\n')
rows_b1 = []
for rnn in ('rnn', 'lstm'):
    for nl in (1, 2, 3):
        for hs in (128, 512):
            for inj in ('preinject', 'initinject'):
                name = f'{rnn}_{inj}_L{nl}_H{hs}'
                r = eval_keras(name)
                if r:
                    r['rnn_type'] = rnn
                    r['injection'] = 'pre' if 'pre' in inj else 'init'
                    r['num_layers'] = nl
                    r['hidden'] = hs
                    rows_b1.append(r)
                    print(f'  {name}: BLEU-4={r["bleu4"]:.4f}')

=== Evaluasi BLEU-4: Pre-inject vs Init-inject ===



I0000 00:00:1778844300.277296    1832 service.cc:152] XLA service 0x4ab9c760 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1778844300.277335    1832 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1778844300.277341    1832 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1778844300.361414    1832 cuda_dnn.cc:529] Loaded cuDNN version 91002


I0000 00:00:1778844301.029042    1832 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


  rnn_preinject_L1_H128: BLEU-4=0.0042


  rnn_initinject_L1_H128: BLEU-4=0.1394


...

  lstm_initinject_L2_H512: BLEU-4=0.1451


  lstm_preinject_L3_H128: BLEU-4=0.0038


  lstm_initinject_L3_H128: BLEU-4=0.1381


  lstm_preinject_L3_H512: BLEU-4=0.0050


  lstm_initinject_L3_H512: BLEU-4=0.1524


In [4]:
if rows_b1:
    df1 = pd.DataFrame(rows_b1)
    for rnn in df1['rnn_type'].unique():
        sub = df1[df1['rnn_type'] == rnn]
        best_pre  = sub[sub['injection']=='pre' ]['bleu4'].max() if (sub['injection']=='pre' ).any() else None
        best_init = sub[sub['injection']=='init']['bleu4'].max() if (sub['injection']=='init').any() else None
        print(f'{rnn.upper():4s} — best pre-inject:  {best_pre}  |  best init-inject: {best_init}')

    n_inj = df1['injection'].nunique()
    if n_inj > 1:
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        for ax, rnn in zip(axes, ['rnn', 'lstm']):
            sub = df1[df1['rnn_type']==rnn].sort_values('model')
            colors = ['#4C72B0' if r=='pre' else '#DD8452' for r in sub['injection']]
            ax.bar(range(len(sub)), sub['bleu4'], color=colors, tick_label=sub['model'].tolist())
            ax.set_title(f'{rnn.upper()} — Pre vs Init-inject BLEU-4')
            ax.set_ylabel('BLEU-4')
            ax.set_xticklabels(sub['model'], rotation=45, ha='right', fontsize=7)
        fig.legend(handles=[Patch(facecolor='#4C72B0', label='Pre-inject'),
                             Patch(facecolor='#DD8452', label='Init-inject')], loc='upper right')
        plt.tight_layout()
        plt.savefig(os.path.join(PLOTS, 'bonus_pre_vs_init_bleu4.png'), dpi=150, bbox_inches='tight')
        plt.show()

    with open(os.path.join(TABLES, 'bonus_pre_vs_init.json'), 'w') as f:
        json.dump(rows_b1, f, indent=2)
    display(df1[['model','injection','bleu4']].sort_values('bleu4', ascending=False))
    print('Bonus 1 selesai.')
else:
    print('Tidak ada model tersedia. Jalankan training terlebih dahulu.')

RNN  — best pre-inject:  0.0051  |  best init-inject: 0.1465
LSTM — best pre-inject:  0.005  |  best init-inject: 0.1524


,model,injection,bleu4
23,lstm_initinject_L3_H512,init,0.1524
15,lstm_initinject_L1_H512,init,0.1500
11,rnn_initinject_L3_H512,init,0.1465
19,lstm_initinject_L2_H512,init,0.1451
7,rnn_initinject_L2_H512,init,0.1434
1,rnn_initinject_L1_H128,init,0.1394
21,lstm_initinject_L3_H128,init,0.1381
3,rnn_initinject_L1_H512,init,0.1345
9,rnn_initinject_L3_H128,init,0.1325
17,lstm_initinject_L2_H128,init,0.1283


Bonus 1 selesai.


---
## Bonus 2 — Beam Search Decoder (k = 3 dan k = 5)

Beam search mempertahankan **k hipotesis terbaik** di setiap langkah decoding, dibanding greedy yang hanya mempertahankan 1. Implementasi ada di `CaptioningPipeline.beam_search_decode()` (`src/rnn/forward_propagation.py`).

In [5]:
# Pilih model terbaik berdasarkan val_loss terkecil
results_path = os.path.join(WEIGHTS, 'training_results_rnn.json')
best_name = None
if os.path.exists(results_path):
    with open(results_path) as f:
        all_results = json.load(f)
    lstm_pre = {k: v for k, v in all_results.items() if 'lstm_preinject' in k}
    if lstm_pre:
        best_name = min(lstm_pre, key=lambda k: min(lstm_pre[k]['history']['val_loss']))

if best_name is None:
    for cand in ['lstm_preinject_L2_H512', 'lstm_preinject_L1_H512',
                 'lstm_preinject_L2_H128', 'lstm_preinject_L1_H128']:
        if os.path.exists(os.path.join(WEIGHTS, f'{cand}.h5')):
            best_name = cand
            break

assert best_name, 'Tidak ada LSTM weight tersedia'
print(f'Model terpilih: {best_name}')

keras_m = keras.models.load_model(
    os.path.join(WEIGHTS, f'{best_name}.h5'),
    compile=False, custom_objects=CUSTOM)
rnn_type_b2 = 'lstm' if 'lstm' in best_name else 'rnn'
if rnn_type_b2 == 'lstm':
    captioner = LSTMCaptioner.from_keras(keras_m, vocab, max_caption_len=30)
else:
    captioner = SimpleRNNCaptioner.from_keras(keras_m, vocab, max_caption_len=30)
print(f'Scratch captioner loaded: {rnn_type_b2.upper()}')

Model terpilih: lstm_preinject_L1_H512


Scratch captioner loaded: LSTM


In [6]:
N_BEAM = min(50, len(test_names))  # subset agar beam search tidak terlalu lama
eval_feats = test_feats[:N_BEAM]
eval_refs  = all_refs[:N_BEAM]
eval_names = test_names[:N_BEAM]
print(f'Mengevaluasi {N_BEAM} gambar...\n')

# Greedy (batch)
t0 = time.perf_counter()
greedy_hyps = captioner.greedy_decode(eval_feats, max_len=30)
t_greedy = time.perf_counter() - t0
bleu_greedy = compute_bleu4(greedy_hyps, eval_refs)
print(f'Greedy          : BLEU-4={bleu_greedy:.4f}  time={t_greedy:.2f}s')

# Beam k=3
t0 = time.perf_counter()
beam3_hyps = [captioner.beam_search_decode(f, beam_width=3, max_len=30) for f in eval_feats]
t_beam3 = time.perf_counter() - t0
bleu_beam3 = compute_bleu4(beam3_hyps, eval_refs)
print(f'Beam search k=3 : BLEU-4={bleu_beam3:.4f}  time={t_beam3:.2f}s')

# Beam k=5
t0 = time.perf_counter()
beam5_hyps = [captioner.beam_search_decode(f, beam_width=5, max_len=30) for f in eval_feats]
t_beam5 = time.perf_counter() - t0
bleu_beam5 = compute_bleu4(beam5_hyps, eval_refs)
print(f'Beam search k=5 : BLEU-4={bleu_beam5:.4f}  time={t_beam5:.2f}s')

Mengevaluasi 50 gambar...



Greedy          : BLEU-4=0.0095  time=0.33s


Beam search k=3 : BLEU-4=0.0075  time=13.35s


Beam search k=5 : BLEU-4=0.0067  time=22.12s


In [7]:
# Contoh caption
print('=== Contoh Caption (5 gambar pertama) ===')
for i in range(min(5, N_BEAM)):
    ref0 = ' '.join(eval_refs[i][0]) if eval_refs[i] else ''
    print(f'\n[{eval_names[i]}]')
    print(f'  Greedy  : {"%s" % " ".join(greedy_hyps[i])}')
    print(f'  Beam k=3: {"%s" % " ".join(beam3_hyps[i])}')
    print(f'  Beam k=5: {"%s" % " ".join(beam5_hyps[i])}')
    print(f'  Ref[0]  : {ref0}')

# Summary table
df2 = pd.DataFrame([
    {'Decoder': 'Greedy',     'BLEU-4': bleu_greedy, 'Time (s)': round(t_greedy, 2)},
    {'Decoder': 'Beam (k=3)', 'BLEU-4': bleu_beam3,  'Time (s)': round(t_beam3,  2)},
    {'Decoder': 'Beam (k=5)', 'BLEU-4': bleu_beam5,  'Time (s)': round(t_beam5,  2)},
])
display(df2)

fig, ax = plt.subplots(figsize=(7, 4))
colors = ['#4C72B0', '#DD8452', '#55A868']
bars = ax.bar(df2['Decoder'], df2['BLEU-4'], color=colors)
ax.set_title(f'Greedy vs Beam Search — BLEU-4 ({N_BEAM} test images)')
ax.set_ylabel('BLEU-4')
for bar, v in zip(bars, df2['BLEU-4']):
    ax.text(bar.get_x() + bar.get_width()/2, v + 0.0002, f'{v:.4f}', ha='center', va='bottom')
plt.tight_layout()
plt.savefig(os.path.join(PLOTS, 'bonus_beam_vs_greedy_bleu4.png'), dpi=150, bbox_inches='tight')
plt.show()

with open(os.path.join(TABLES, 'bonus_beam_search.json'), 'w') as f:
    json.dump(df2.to_dict('records'), f, indent=2)
print('Bonus 2 selesai.')

=== Contoh Caption (5 gambar pertama) ===

[429270993_294ba8e64c.jpg]
  Greedy  : man a and woman in of standing a of
  Beam k=3: man a and woman a in of standing a of in of
  Beam k=5: man a and woman standing a in of street a and woman a
  Ref[0]  : a man in camouflage pants and a backwards cap carrying a toolbox and a backpack .

[3439982121_0afc6d5973.jpg]
  Greedy  : white white white white white white black white black white black white black white black white black white white black white white black white white black white white black white
  Beam k=3: white white white white black white white black white black white black white black white black white black white black white black white black white black white black white
  Beam k=5: white white white white white black white black white black white black white black white black white black white black white black white black white
  Ref[0]  : an elderly person holds a white doge and kisses their cheek .

[3274375509_4fe91a94c0

,Decoder,BLEU-4,Time (s)
0,Greedy,0.009531,0.33
1,Beam (k=3),0.007542,13.35
2,Beam (k=5),0.006724,22.12


Bonus 2 selesai.


---
## Bonus 3 — Batch Inference

Seluruh forward propagation from scratch mendukung **batch_size > 1**. Semua operasi layer (Embedding, SimpleRNNCell, LSTMCell, DenseProjection, DenseOutput) beroperasi pada dimensi batch *(N, ...)*, dan `greedy_decode()` menerima parameter `batch_size` implisit melalui jumlah baris input.

In [8]:
print('=== Bonus 3: Batch Inference ===\n')

# 1. Verifikasi: hasil batch identik dengan sequential
N_CHECK = 8
feats_check = test_feats[:N_CHECK]

batch_caps = captioner.greedy_decode(feats_check, max_len=30)
seq_caps   = [captioner.greedy_decode(feats_check[i:i+1], max_len=30)[0] for i in range(N_CHECK)]

match = all(b == s for b, s in zip(batch_caps, seq_caps))
print(f'Batch == Sequential (N={N_CHECK}): {"PASS" if match else "FAIL"}')
for i in range(3):
    print(f'  [{i}] batch: {"%s" % " ".join(batch_caps[i])}')
    print(f'  [{i}] seq  : {"%s" % " ".join(seq_caps[i])}')

# 2. Speed comparison across batch sizes
print('\n=== Speed by batch_size ===')
rows_b3 = []
for bs in [1, 8, 32, 64, len(test_names)]:
    if bs > len(test_feats): continue
    sample = test_feats[:bs]
    captioner.greedy_decode(sample[:1], max_len=30)  # warmup
    t0 = time.perf_counter()
    captioner.greedy_decode(sample, max_len=30)
    elapsed = time.perf_counter() - t0
    ms_per = elapsed / bs * 1000
    rows_b3.append({'batch_size': bs, 'total_s': round(elapsed, 3), 'ms_per_image': round(ms_per, 2)})
    print(f'  batch_size={bs:5d}: {elapsed:.3f}s total  |  {ms_per:.2f} ms/image')

df3 = pd.DataFrame(rows_b3)
display(df3)
with open(os.path.join(TABLES, 'bonus_batch_inference.json'), 'w') as f:
    json.dump(rows_b3, f, indent=2)
print('Bonus 3 selesai.')

=== Bonus 3: Batch Inference ===



Batch == Sequential (N=8): PASS
  [0] batch: man a and woman in of standing a of
  [0] seq  : man a and woman in of standing a of
  [1] batch: white white white white white white black white black white black white black white black white black white white black white white black white white black white white black white
  [1] seq  : white white white white white white black white black white black white black white black white black white white black white white black white white black white white black white
  [2] batch: woman a and woman a girl a and woman a girl a and woman a
  [2] seq  : woman a and woman a girl a and woman a girl a and woman a

=== Speed by batch_size ===
  batch_size=    1: 0.031s total  |  30.65 ms/image


  batch_size=    8: 0.149s total  |  18.63 ms/image


  batch_size=   32: 0.235s total  |  7.33 ms/image


  batch_size=   64: 0.348s total  |  5.43 ms/image


  batch_size= 1012: 3.699s total  |  3.65 ms/image


,batch_size,total_s,ms_per_image
0,1,0.031,30.65
1,8,0.149,18.63
2,32,0.235,7.33
3,64,0.348,5.43
4,1012,3.699,3.65


Bonus 3 selesai.


---
## Bonus 4 — Backward Propagation

Implementasi backward propagation from scratch untuk semua layer RNN/LSTM:
- `Embedding.backward()` — gradient w.r.t. embedding matrix
- `SimpleRNNCell.backward()` — BPTT (Back-Propagation Through Time)
- `LSTMCell.backward()` — BPTT dengan 4 gate (i, f, g, o)

Verifikasi menggunakan **numerical gradient check** (finite differences, ε=1e-4).

In [9]:
def num_grad(loss_fn, weight, eps=1e-4):
    """Numerical gradient via centered finite differences (in-place perturbation)."""
    grad = np.zeros_like(weight)
    for idx in np.ndindex(weight.shape):
        orig = weight[idx]
        weight[idx] = orig + eps;  lp = loss_fn()
        weight[idx] = orig - eps;  lm = loss_fn()
        weight[idx] = orig
        grad[idx] = (lp - lm) / (2 * eps)
    return grad

def rel_err(num, ana):
    return float(np.max(np.abs(num - ana)) / (np.max(np.abs(num)) + 1e-8))

print('Utility functions defined.')

Utility functions defined.


In [10]:
print('=== SimpleRNNCell — Numerical Gradient Check ===\n')
np.random.seed(42)
N_, T_, D_, H_ = 2, 4, 8, 16

x_rnn = np.random.randn(N_, T_, D_).astype(np.float64)
h0_rnn = np.zeros((N_, H_), dtype=np.float64)
d_rnn  = np.random.randn(N_, H_).astype(np.float64)

rnn = SimpleRNNCell(
    W_xh=np.random.randn(D_, H_).astype(np.float64) * 0.1,
    W_hh=np.random.randn(H_, H_).astype(np.float64) * 0.1,
    b_h =np.zeros(H_, dtype=np.float64),
)

# Analytical gradients
hs_rnn, _ = rnn.forward(x_rnn, h0_rnn, return_sequences=True)
_, grads_rnn = rnn.backward(d_rnn, x_rnn, hs_rnn, h0_rnn)

# Loss: sum(h_final * d_rnn)
def rnn_loss(): 
    out, _ = rnn.forward(x_rnn, h0_rnn, return_sequences=False)
    return float(np.sum(out * d_rnn))

results_b4 = []
for key, weight in [('W_xh', rnn.W_xh), ('W_hh', rnn.W_hh), ('b_h', rnn.b_h)]:
    ng = num_grad(rnn_loss, weight)
    err = rel_err(ng, grads_rnn[key])
    status = 'PASS' if err < 1e-3 else 'FAIL'
    print(f'  SimpleRNN {key:6s}: rel_err={err:.2e}  [{status}]')
    results_b4.append({'layer': 'SimpleRNN', 'param': key, 'rel_err': round(err, 8), 'status': status})

print()

=== SimpleRNNCell — Numerical Gradient Check ===

  SimpleRNN W_xh  : rel_err=6.87e-09  [PASS]
  SimpleRNN W_hh  : rel_err=5.63e-09  [PASS]
  SimpleRNN b_h   : rel_err=3.41e-09  [PASS]



In [11]:
print('=== LSTMCell — Numerical Gradient Check ===\n')
np.random.seed(99)

x_lstm  = np.random.randn(N_, T_, D_).astype(np.float64)
h0_lstm = np.zeros((N_, H_), dtype=np.float64)
c0_lstm = np.zeros((N_, H_), dtype=np.float64)
d_lstm  = np.random.randn(N_, H_).astype(np.float64)

lstm = LSTMCell(
    W_x=np.random.randn(D_, 4*H_).astype(np.float64) * 0.1,
    W_h=np.random.randn(H_, 4*H_).astype(np.float64) * 0.1,
    b  =np.zeros(4*H_, dtype=np.float64),
)

# Get hs and cs via manual forward (forward() doesn't return cs)
h, c = h0_lstm.copy(), c0_lstm.copy()
hs_lstm = np.zeros((N_, T_, H_))
cs_lstm = np.zeros((N_, T_, H_))
for t in range(T_):
    h, c = lstm.step(x_lstm[:, t, :], h, c)
    hs_lstm[:, t, :] = h
    cs_lstm[:, t, :] = c

_, grads_lstm = lstm.backward(d_lstm, x_lstm, hs_lstm, cs_lstm, h0_lstm, c0_lstm)

def lstm_loss():
    h_, c_ = h0_lstm.copy(), c0_lstm.copy()
    for t in range(T_):
        h_, c_ = lstm.step(x_lstm[:, t, :], h_, c_)
    return float(np.sum(h_ * d_lstm))

for key, weight in [('W_x', lstm.W_x), ('W_h', lstm.W_h), ('b', lstm.b)]:
    ng = num_grad(lstm_loss, weight)
    err = rel_err(ng, grads_lstm[key])
    status = 'PASS' if err < 1e-3 else 'FAIL'
    print(f'  LSTM {key:3s}: rel_err={err:.2e}  [{status}]')
    results_b4.append({'layer': 'LSTM', 'param': key, 'rel_err': round(err, 8), 'status': status})

print('\nBonus 4 selesai.')

=== LSTMCell — Numerical Gradient Check ===



  LSTM W_x: rel_err=1.66e-08  [PASS]


  LSTM W_h: rel_err=3.93e-10  [PASS]
  LSTM b  : rel_err=4.90e-09  [PASS]

Bonus 4 selesai.


In [12]:
print('\n' + '='*60)
print('RINGKASAN BONUS RNN/LSTM')
print('='*60)
print('[Bonus 1] Init-inject vs Pre-inject  : implementasi di src/rnn/models.py')
print('          build_rnn_decoder_init_inject() — image → h0/c0 (initial state)')
print('[Bonus 2] Beam Search Decoder         : implementasi di src/rnn/forward_propagation.py')
print('          CaptioningPipeline.beam_search_decode(k=3/5)')
print('[Bonus 3] Batch Inference             : greedy_decode(feats, max_len) — batch dim N')
print('          Semua layer beroperasi pada dimensi (N, ...)')
print('[Bonus 4] Backward Propagation        : implementasi di src/rnn/layers.py')
print('          SimpleRNNCell.backward() + LSTMCell.backward() (BPTT)')

if results_b4:
    df4 = pd.DataFrame(results_b4)
    print('\n--- Numerical Gradient Check ---')
    display(df4)
    with open(os.path.join(TABLES, 'bonus_backprop_check.json'), 'w') as f:
        json.dump(results_b4, f, indent=2)
print('\nBonus demo selesai. Plot: results/plots/  Tables: results/tables/')


RINGKASAN BONUS RNN/LSTM
[Bonus 1] Init-inject vs Pre-inject  : implementasi di src/rnn/models.py
          build_rnn_decoder_init_inject() — image → h0/c0 (initial state)
[Bonus 2] Beam Search Decoder         : implementasi di src/rnn/forward_propagation.py
          CaptioningPipeline.beam_search_decode(k=3/5)
[Bonus 3] Batch Inference             : greedy_decode(feats, max_len) — batch dim N
          Semua layer beroperasi pada dimensi (N, ...)
[Bonus 4] Backward Propagation        : implementasi di src/rnn/layers.py
          SimpleRNNCell.backward() + LSTMCell.backward() (BPTT)

--- Numerical Gradient Check ---


,layer,param,rel_err,status
0,SimpleRNN,W_xh,1.000000e-08,PASS
1,SimpleRNN,W_hh,1.000000e-08,PASS
2,SimpleRNN,b_h,0.000000e+00,PASS
3,LSTM,W_x,2.000000e-08,PASS
4,LSTM,W_h,0.000000e+00,PASS
5,LSTM,b,0.000000e+00,PASS



Bonus demo selesai. Plot: results/plots/  Tables: results/tables/
